# 03 · P2 训练矩阵没有开始（HANDOFF §6 P2）

**对应 HANDOFF §6 P2**「R1–R4 × seed 真训练（train G12D）」，
以及 §5「每个 run 建议 ≥3–5 seed」。

**一个 run 都没跑。** 但挡住它的不是编排——编排已经就绪——
而是两件更前面的事。


In [1]:
%matplotlib inline
import json, warnings
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
plt.rcParams.update({
    "figure.dpi": 300, "savefig.dpi": 300, "font.size": 9,
    "axes.grid": True, "grid.alpha": .25,
    "axes.spines.top": False, "axes.spines.right": False,
})
R = Path("/lus/lfs1aip2/projects/public/u6gb/tasks/large-discovery-model/ldm_rl/results")
def load(name): return json.loads((R / name).read_text())

ready = pd.DataFrame([
 ["kernel 选定", "就绪", "定 fp。sk 的阶是 2.68，生产规模跑不完（见 05）"],
 ["双节点分卡布局", "就绪", "needs_offload=False，绕开 torch_memory_saver 断言"],
 ["跨分配起 9B", "就绪", "JOB_A/JOB_B 分别指定；空节点几乎总是跨分配出现"],
 ["每 run 独立 GP / seed / 输出目录", "就绪", "烘焙进各自的 episodes.jsonl"],
 ["编排活过会话", "就绪", "改用 tmux；setsid 保护不了 Slurm step"],
 ["排除自己启动窗口里的节点", "就绪", "靠 squeue -s，显存看不见启动中的 run"],
 ["主机内存预检", "就绪", "读 job cgroup 余量，门槛 300 GB"],
 ["reward 有效", "**阻塞**", "reward 恒零，跑完也学不到东西（见 01）"],
 ["9B 能活过启动", "**阻塞**", "29 个 9B run 零存活（见 02）"],
 ["16 个空节点", "**缺**", "空节点零星出现，凑不齐 8 对"],
], columns=["前置条件", "状态", "依据"])
ready.style.hide(axis="index")

前置条件,状态,依据
kernel 选定,就绪,定 fp。sk 的阶是 2.68，生产规模跑不完（见 05）
双节点分卡布局,就绪,needs_offload=False，绕开 torch_memory_saver 断言
跨分配起 9B,就绪,JOB_A/JOB_B 分别指定；空节点几乎总是跨分配出现
每 run 独立 GP / seed / 输出目录,就绪,烘焙进各自的 episodes.jsonl
编排活过会话,就绪,改用 tmux；setsid 保护不了 Slurm step
排除自己启动窗口里的节点,就绪,靠 squeue -s，显存看不见启动中的 run
主机内存预检,就绪,读 job cgroup 余量，门槛 300 GB
reward 有效,**阻塞**,reward 恒零，跑完也学不到东西（见 01）
9B 能活过启动,**阻塞**,29 个 9B run 零存活（见 02）
16 个空节点,**缺**,空节点零星出现，凑不齐 8 对


**读法**：七项技术准备已经完成，**挡着 P2 的是最后三项**。

其中前两项是致命的：reward 恒零意味着 16 个 run × 十几小时**全部拿零梯度**，
而 9B 零存活意味着 run 根本活不到产出结果。**在这两项解决之前跑 P2，
消耗的机时不会换来任何可用结论。**

## 矩阵的规模与代价

In [2]:
m = pd.DataFrame([
 ["配置数", "4", "R1–R4"],
 ["种子数", "4", "HANDOFF §5 建议 ≥3–5；取 4 让每波正好填满 64 卡"],
 ["run 总数", "16", "4 × 4"],
 ["每 run 节点", "2", "actor 4 卡 + sglang 4 卡（双节点分卡）"],
 ["同时需要的节点", "16", "一波 8 个 run"],
 ["波数", "2", "16 run ÷ 8"],
 ["每 run env.step 次数", "4000", "50 步 × 4 条轨迹 × 20 轮"],
], columns=["项", "值", "说明"])
m.style.hide(axis="index")

项,值,说明
配置数,4,R1–R4
种子数,4,HANDOFF §5 建议 ≥3–5；取 4 让每波正好填满 64 卡
run 总数,16,4 × 4
每 run 节点,2,actor 4 卡 + sglang 4 卡（双节点分卡）
同时需要的节点,16,一波 8 个 run
波数,2,16 run ÷ 8
每 run env.step 次数,4000,50 步 × 4 条轨迹 × 20 轮


## `--seed-offset` 不存在

HANDOFF §5 写「每个 run 建议 ≥3–5 seed(`--seed-offset`)」，
但**这个参数在 slime 的 `arguments.py` 里不存在**，只有 `--seed`（默认 1234）。
详见 [06_seed_offset_missing.ipynb](06_seed_offset_missing.ipynb)。

环境侧真正的 seed 烘焙在每条 episode 的 prompt JSON 里，
只能靠重新生成 episodes 文件来改——编排脚本已经这么做了。